<a href="https://colab.research.google.com/github/EgorNezo/-/blob/add-name-Dasha/Variant_12_Garmoniki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.fft import fft, fftfreq
import os

# ======================================================
# 1. МОНТИРУЕМ GOOGLE DRIVE
# ======================================================
from google.colab import drive
drive.mount('/content/drive')

# ======================================================
# 2. ЧИТАЕМ ФАЙЛ ПРАВИЛЬНО
# ======================================================
csv_path = '/content/drive/MyDrive/Current.csv'

print("Читаем файл Current.csv...")

# Пробуем разные варианты чтения
df = None

# Вариант 1: без заголовка, разделитель - запятая
try:
    df = pd.read_csv(csv_path, header=None, sep=',')
    print("   Вариант 1: без заголовка, разделитель ',' -> успешно")
except:
    pass

# Вариант 2: без заголовка, разделитель - точка с запятой
if df is None or df.shape[1] < 2:
    try:
        df = pd.read_csv(csv_path, header=None, sep=';')
        print("   Вариант 2: без заголовка, разделитель ';' -> успешно")
    except:
        pass

# Вариант 3: без заголовка, разделитель - пробел
if df is None or df.shape[1] < 2:
    try:
        df = pd.read_csv(csv_path, header=None, sep='\s+')
        print("   Вариант 3: без заголовка, разделитель 'пробел' -> успешно")
    except:
        pass

if df is None:
    raise Exception("Не удалось прочитать файл")

print(f"\n   Столбцов: {df.shape[1]}, строк: {df.shape[0]}")

# ======================================================
# 3. РАЗБИРАЕМСЯ С ДАННЫМИ
# ======================================================
# Если получился 1 столбец - скорее всего данные идут подряд:
# столбец1 = время, столбец2 = ток, столбец3 = напряжение и т.д.
# Нужно понять структуру. Попробуем определить автоматически

print(f"\nПервые значения (для определения структуры):")
print(df.head(10))

# Если столбец один, то, вероятно, все данные перемешаны
if df.shape[1] == 1:
    # Пробуем понять: может быть каждые 2 или 3 значения - это отдельный сигнал
    data_1d = df[0].values

    # Похоже на осциллограмму: сигналы идут друг за другом
    # В типичном осциллографе: 1 канал (ток), 2 канал (напряжение)
    # Попробуем разделить на 2 или 3 сигнала

    # Проверяем гипотезу: каждый второй элемент - отдельный сигнал
    # Например, чётные - ток, нечётные - напряжение
    n_signals = 2  # предположим ток и напряжение

    # Длина каждого сигнала
    signal_len = len(data_1d) // n_signals

    if signal_len > 100:
        # Создаём DataFrame с колонками
        data_reshaped = data_1d[:signal_len * n_signals].reshape(n_signals, signal_len).T
        df_new = pd.DataFrame(data_reshaped, columns=['Ток', 'Напряжение'])
        df = df_new
        print(f"\n   Разделили на {n_signals} сигнала: {df.columns.tolist()}")
        print(f"   Длина каждого сигнала: {signal_len} точек")

print(f"\nФинальная структура: {df.shape} столбцов: {df.columns.tolist()}")

# ======================================================
# 4. ВЫВОД ПЕРВЫХ И ПОСЛЕДНИХ СТРОК
# ======================================================
print("\n" + "="*60)
print("ПЕРВЫЕ 5 СТРОК:")
print(df.head())

print("\nПОСЛЕДНИЕ 5 СТРОК:")
print(df.tail())
print("="*60)

# ======================================================
# 5. ПОДГОТОВКА ДАННЫХ
# ======================================================
# Если нет отдельного столбца времени - создаём его
if 'время' not in df.columns.str.lower().tolist():
    # Создаём время с шагом 0.00015625 секунд (0.15625 мс)
    dt_sample = 0.00015625  # 0.15625 мс в секундах
    time = np.arange(len(df)) * dt_sample
    print(f"\nСоздали время: {len(time)} точек, шаг {dt_sample*1000:.6f} мс")
else:
    time_col = df.columns[df.columns.str.lower() == 'время'][0]
    time = df[time_col].values

# Сигналы (все столбцы кроме времени)
signal_cols = [c for c in df.columns if 'время' not in c.lower()]
print(f"Сигналы для анализа: {signal_cols}")

# ======================================================
# 6. ПОСТРОЕНИЕ ГРАФИКОВ
# ======================================================
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Временной ряд',
                    'Амплитудно-частотный спектр'),
    vertical_spacing=0.15
)

# Временной график
for col in signal_cols:
    fig.add_trace(
        go.Scattergl(x=time, y=df[col].values,
                     mode='lines',
                     name=col,
                     line=dict(width=1)),
        row=1, col=1
    )

# Спектр для первого сигнала
signal_name = signal_cols[0]
signal_data = df[signal_name].values.astype(float)

# Убираем NaN
signal_data = np.nan_to_num(signal_data)

N = len(signal_data)
yf = fft(signal_data)
dt = time[1] - time[0] if len(time) > 1 else 0.00015625
xf = fftfreq(N, dt)[:N//2]
magnitude = (2.0/N) * np.abs(yf[:N//2])

# Ограничиваем до 1000 Гц
max_freq = 1000
max_idx = np.where(xf <= max_freq)[0]
if len(max_idx) > 0:
    max_idx = max_idx[-1]
else:
    max_idx = len(xf)

fig.add_trace(
    go.Scattergl(x=xf[:max_idx], y=magnitude[:max_idx],
                 mode='lines',
                 name=f'Спектр {signal_name}',
                 line=dict(color='red', width=1.5)),
    row=2, col=1
)

fig.update_xaxes(title_text="Время (с)", row=1, col=1)
fig.update_yaxes(title_text="Амплитуда (А/В)", row=1, col=1)
fig.update_xaxes(title_text="Частота (Гц)", row=2, col=1)
fig.update_yaxes(title_text="Амплитуда", row=2, col=1)

fig.update_layout(height=800, title_text="Вариант 12: Анализ тока и напряжения")
fig.show()

# ======================================================
# 7. АНАЛИЗ ГАРМОНИК
# ======================================================
print("\n" + "="*60)
print("АНАЛИЗ ВЫСШИХ ГАРМОНИК (база 50 Гц)")
print("="*60)

base_freq = 50.0
tolerance = 3.0

# Находим пик на 50 Гц (или ближайший)
idx_50 = np.argmin(np.abs(xf[:max_idx] - 50))
amp_50 = magnitude[idx_50] if idx_50 < len(magnitude) else 0
print(f"Амплитуда на частоте {xf[idx_50]:.2f} Гц (ближайшая к 50 Гц): {amp_50:.6f}")

harmonics_found = []

for h in range(2, 11):
    target = h * base_freq
    idx = np.argmin(np.abs(xf[:max_idx] - target))
    if idx >= len(magnitude):
        continue
    nearest_freq = xf[idx]
    amp = magnitude[idx]

    if abs(nearest_freq - target) < tolerance and amp > amp_50 * 0.01:
        harmonics_found.append(h)
        print(f"Гармоника {h:2d} ({target} Гц): {nearest_freq:.2f} Гц, амплитуда {amp:.6f} -> ЕСТЬ")
    else:
        if amp > amp_50 * 0.005:
            print(f"Гармоника {h:2d} ({target} Гц): частота {nearest_freq:.2f} Гц, амплитуда {amp:.6f} -> НЕТ (порог)")
        else:
            print(f"Гармоника {h:2d} ({target} Гц): не обнаружена")

# ======================================================
# 8. ВЫВОД
# ======================================================
print("\n" + "="*60)
print("ВЫВОД ПО ЗАДАНИЮ (Таблица 2 - АЧХ и высшие гармоники)")
print("="*60)

if len(harmonics_found) > 0:
    print(f"✓ ВЫСШИЕ ГАРМОНИКИ ПРИСУТСТВУЮТ")
    print(f"  Обнаружены: {', '.join([str(h) for h in harmonics_found])}-я гармоники")
    print("  Сигнал имеет искажения (нелинейная нагрузка).")
else:
    print("✓ ВЫСШИЕ ГАРМОНИКИ ОТСУТСТВУЮТ")
    print("  Сигнал близок к чистой синусоиде 50 Гц.")
print("="*60)

# ======================================================
# 9. СОХРАНЕНИЕ
# ======================================================
os.makedirs("variant_12_output", exist_ok=True)

fig.write_html("variant_12_output/graph.html")
print(f"\nСохранено: variant_12_output/graph.html")

# Excel
spectrum_df = pd.DataFrame({
    'Частота_Гц': xf[:max_idx],
    'Амплитуда': magnitude[:max_idx]
})

with pd.ExcelWriter("variant_12_output/data.xlsx") as writer:
    df.to_excel(writer, sheet_name='Данные', index=False)
    spectrum_df.to_excel(writer, sheet_name='Спектр', index=False)

print("Сохранено: variant_12_output/data.xlsx")

print("\n" + "="*60)
print("ГОТОВО! Загрузи файлы на GitHub:")
print("- Сохрани блокнот: Файл -> Сохранить копию на GitHub")
print("- Или скачай .ipynb и залей в репозиторий")
print("="*60)

<>:43: SyntaxWarning:

invalid escape sequence '\s'

<>:43: SyntaxWarning:

invalid escape sequence '\s'

/tmp/ipykernel_2787/3072467461.py:43: SyntaxWarning:

invalid escape sequence '\s'



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Читаем файл Current.csv...
   Вариант 1: без заголовка, разделитель ',' -> успешно
   Вариант 2: без заголовка, разделитель ';' -> успешно
   Вариант 3: без заголовка, разделитель 'пробел' -> успешно

   Столбцов: 1, строк: 60287

Первые значения (для определения структуры):
       0
0   2200
1   7700
2  12100
3  17600
4  23100
5  28600
6  33000
7  37400
8  41800
9  46200

   Разделили на 2 сигнала: ['Ток', 'Напряжение']
   Длина каждого сигнала: 30143 точек

Финальная структура: (30143, 2) столбцов: ['Ток', 'Напряжение']

ПЕРВЫЕ 5 СТРОК:
     Ток  Напряжение
0   2200        3300
1   7700       -1100
2  12100       -6600
3  17600      -11000
4  23100      -16500

ПОСЛЕДНИЕ 5 СТРОК:
         Ток  Напряжение
30138  27500      -27500
30139  23100      -23100
30140  17600      -18700
30141  14300      -14300
30142   8800      -11000

Создали время: 30143 точек, ш


АНАЛИЗ ВЫСШИХ ГАРМОНИК (база 50 Гц)
Амплитуда на частоте 49.90 Гц (ближайшая к 50 Гц): 59768.752171
Гармоника  2 (100.0 Гц): не обнаружена
Гармоника  3 (150.0 Гц): частота 149.90 Гц, амплитуда 404.077339 -> НЕТ (порог)
Гармоника  4 (200.0 Гц): не обнаружена
Гармоника  5 (250.0 Гц): 249.90 Гц, амплитуда 1265.263514 -> ЕСТЬ
Гармоника  6 (300.0 Гц): не обнаружена
Гармоника  7 (350.0 Гц): 349.91 Гц, амплитуда 620.957051 -> ЕСТЬ
Гармоника  8 (400.0 Гц): не обнаружена
Гармоника  9 (450.0 Гц): не обнаружена
Гармоника 10 (500.0 Гц): не обнаружена

ВЫВОД ПО ЗАДАНИЮ (Таблица 2 - АЧХ и высшие гармоники)
✓ ВЫСШИЕ ГАРМОНИКИ ПРИСУТСТВУЮТ
  Обнаружены: 5, 7-я гармоники
  Сигнал имеет искажения (нелинейная нагрузка).

Сохранено: variant_12_output/graph.html
Сохранено: variant_12_output/data.xlsx

ГОТОВО! Загрузи файлы на GitHub:
- Сохрани блокнот: Файл -> Сохранить копию на GitHub
- Или скачай .ipynb и залей в репозиторий
